In [1]:
import pandas as pd
from google.cloud import storage, bigquery, bigquery_storage
import io

In [2]:
project_id = "valorant-project-2026"
location = "asia-southeast2"
bucket_name = "valorant-raw-data"
client = bigquery.Client(project=project_id)

In [3]:
def create_dataset(client, project_id, dataset_name, location):
    dataset_id = f"{project_id}.{dataset_name}"

    dataset = bigquery.Dataset(dataset_id)
    dataset.location = location

    dataset = client.create_dataset(
        dataset,
        exists_ok=True
    )

    print(f"Dataset ready: {dataset_id}")

In [4]:
layer = ["bronze", "silver", "gold", "dbt_ci", "dbt_dev", "dbt_prod"]
for i in layer:
    create_dataset(
        client, project_id, i, location
    )

Dataset ready: valorant-project-2026.bronze
Dataset ready: valorant-project-2026.silver
Dataset ready: valorant-project-2026.gold
Dataset ready: valorant-project-2026.dbt_ci
Dataset ready: valorant-project-2026.dbt_dev
Dataset ready: valorant-project-2026.dbt_prod


In [3]:
def load_table(client, project_id, table, bucket_name):
    table_id = f"{project_id}.bronze.{table}"
    gcs_uri = f"gs://{bucket_name}/raw/{table}.parquet"

    job_config = bigquery.LoadJobConfig(
        autodetect=True,
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    print(f"Starting load job for {gcs_uri} ...")

    load_job = client.load_table_from_uri(
        gcs_uri, table_id, job_config=job_config
    )

    load_job.result()

    destination_table = client.get_table(table_id)
    print(f"Success! Loaded {destination_table.num_rows} rows into {table_id}")

In [4]:
table = ["tours", "matches", "games_overview", "games_economy", "map_vetos", "players"]
for i in table:
    load_table(
        client, project_id, i, bucket_name
    )

Starting load job for gs://valorant-raw-data/raw/tours.parquet ...
Success! Loaded 48 rows into valorant-project-2026.bronze.tours
Starting load job for gs://valorant-raw-data/raw/matches.parquet ...
Success! Loaded 1259 rows into valorant-project-2026.bronze.matches
Starting load job for gs://valorant-raw-data/raw/games_overview.parquet ...
Success! Loaded 3224 rows into valorant-project-2026.bronze.games_overview
Starting load job for gs://valorant-raw-data/raw/games_economy.parquet ...
Success! Loaded 3941 rows into valorant-project-2026.bronze.games_economy
Starting load job for gs://valorant-raw-data/raw/map_vetos.parquet ...
Success! Loaded 8814 rows into valorant-project-2026.bronze.map_vetos
Starting load job for gs://valorant-raw-data/raw/players.parquet ...
Success! Loaded 96720 rows into valorant-project-2026.bronze.players


In [ ]:
# Duplicated and Missing Value Check
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT tour_id) AS unique_value,
    COUNTIF(tour_id IS NULL) AS null_tour_id,
FROM `{project_id}.bronze.tours`
"""

result = client.query(query).result()
for row in result:
    print(dict(row))

{'total_rows': 48, 'unique_value': 48, 'null_tour_id': 0}


In [33]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.dims_tours`
AS

SELECT
    tour_id,
    tour_name,
    tour_tag,

    CASE
        WHEN tour_stage NOT IN (
            'Stage 1',
            'Stage 2',
            'Champions'
        )
        THEN 'Masters'
        ELSE tour_stage
    END AS tour_stage,

    CASE
        WHEN tour_region = 'Masters'
        THEN 'World'
        ELSE tour_region
    END AS tour_region,

    tour_status
FROM `{project_id}.bronze.tours`;
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.dims_tours")

Silver table created: valorant-project-2026.silver.dims_tours


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.matches`
PARTITION BY match_date
AS

SELECT
    tour_id,
    match_id,

    SAFE.PARSE_DATETIME(
        '%Y-%m-%d %H:%M:%S',
        date
    ) AS match_datetime,

    DATE(
        SAFE.PARSE_DATETIME(
           '%Y-%m-%d %H:%M:%S',
            date 
        )
    ) AS match_date,
    
    bracket,
    home_name,
    home_alias,
    away_name,
    away_alias,
    bo,
    patch,
    home_score,
    away_score,
    home_h2h_win,
    away_h2h_win,
    home_h2h_score,
    away_h2h_score,
    home_n_last_match_win,
    away_n_last_match_win,
    home_n_last_match,
    away_n_last_match,
    home_n_last_match_win / NULLIF(home_n_last_match, 0) AS home_n_last_wr,
    away_n_last_match_win / NULLIF(away_n_last_match, 0) AS away_n_last_wr,
    
FROM `{project_id}.bronze.matches`
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.matches")

Silver table created: valorant-project-2026.silver.partitioned_matches


In [67]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.games_overview`
AS

SELECT
    o.match_id,
    game_id,
    m.match_date,
    m.match_datetime,

    NULLIF(game_map, '-') AS game_map,

    CASE
        WHEN LENGTH(game_duration) = 5
        THEN TIME_DIFF(PARSE_TIME('%M:%S', game_duration), '00:00:00', SECOND)
        WHEN LENGTH(game_duration) = 8
        THEN TIME_DIFF(PARSE_TIME('%H:%M:%S', game_duration), '00:00:00', SECOND)
        ELSE NULL
    END AS game_duration,

    o.home_score,
    o.away_score,
    SAFE_CAST(home_atk_score AS INT64) AS home_atk_score,
    SAFE_CAST(away_atk_score AS INT64) AS away_atk_score,
    SAFE_CAST(home_def_score AS INT64) AS home_def_score,
    SAFE_CAST(away_def_score AS INT64) AS away_def_score,
    SAFE_CAST(home_ot_score AS INT64) AS home_ot_score,
    SAFE_CAST(away_ot_score AS INT64) AS away_ot_score
FROM `{project_id}.bronze.games_overview`o
JOIN `{project_id}.silver.matches` m
ON o.match_id = m.match_id
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.games_overview")

Silver table created: valorant-project-2026.silver.games_overview


In [68]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.games_economy`
AS

SELECT
    e.match_id,
    game_id,
    m.match_date,
    m.match_datetime,

    SAFE_CAST(home_pstl_win AS INT64) AS home_pstl_win,
    SAFE_CAST(away_pstl_win AS INT64) AS away_pstl_win,
    SAFE_CAST(home_eco_round AS INT64) AS home_eco_round,
    SAFE_CAST(away_eco_round AS INT64) AS away_eco_round,
    SAFE_CAST(home_eco_win AS INT64) AS home_eco_win,
    SAFE_CAST(away_eco_win AS INT64) AS away_eco_win,
    SAFE_CAST(home_semi_eco_round AS INT64) AS home_semi_eco_round,
    SAFE_CAST(away_semi_eco_round AS INT64) AS away_semi_eco_round,
    SAFE_CAST(home_semi_eco_win AS INT64) AS home_semi_eco_win,
    SAFE_CAST(away_semi_eco_win AS INT64) AS away_semi_eco_win,
    SAFE_CAST(home_semi_buy_round AS INT64) AS home_semi_buy_round,
    SAFE_CAST(away_semi_buy_round AS INT64) AS away_semi_buy_round,
    SAFE_CAST(home_semi_buy_win AS INT64) AS home_semi_buy_win,
    SAFE_CAST(away_semi_buy_win AS INT64) AS away_semi_buy_win,
    SAFE_CAST(home_full_buy_round AS INT64) AS home_full_buy_round,
    SAFE_CAST(away_full_buy_round AS INT64) AS away_full_buy_round,
    SAFE_CAST(home_full_buy_win AS INT64) AS home_full_buy_win,
    SAFE_CAST(away_full_buy_win AS INT64) AS away_full_buy_win,
FROM `{project_id}.bronze.games_economy` e
JOIN `{project_id}.silver.matches` m
ON m.match_id = e.match_id
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.games_economy")

Silver table created: valorant-project-2026.silver.games_economy


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.players_performance`
AS

SELECT
    game_id,
    name AS player_name,
    team_alias,
    nationality,
    agent,
    mod,
    r,
    acs,
    k,
    d,
    a,
    kd,
    kast,
    adr,
    hs,
    fk,
    fd,
    fkfd
FROM `{project_id}.bronze.players`
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.players_performance")

Silver table created: valorant-project-2026.silver.players_performance


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.map_vetos`
AS

SELECT
    match_id,
    map_name,
    team_name,
    action
FROM `{project_id}.bronze.map_vetos`
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.map_vetos")

Silver table created: valorant-project-2026.silver.map_vetos


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.dims_teams`
AS

SELECT ROW_NUMBER() OVER (ORDER BY team_name) AS team_id, team_name, team_alias, team_region
FROM(
    SELECT DISTINCT
        home_name as team_name,
        home_alias as team_alias,
        t.tour_region as team_region,
    FROM `{project_id}.silver.matches` m
    JOIN `{project_id}.silver.dims_tours` t
    ON m.tour_id = t.tour_id
)
WHERE team_region != 'World'
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.dims_teams")

Silver table created: valorant-project-2026.silver.dims_teams


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.dims_maps`
AS

SELECT ROW_NUMBER() OVER (ORDER BY map_name) AS map_id, map_name
FROM(
    SELECT DISTINCT map_name
    FROM `{project_id}.silver.map_vetos`
)
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.dims_maps")

Silver table created: valorant-project-2026.silver.dims_maps


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.dims_players`
AS

SELECT ROW_NUMBER() OVER (ORDER BY player_name) AS player_id, player_name, player_nationality
FROM(
    SELECT DISTINCT
        player_name,
        nationality as player_nationality
    FROM `{project_id}.silver.players_performance`
)
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.dims_players")

Silver table created: valorant-project-2026.silver.dims_players


In [7]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.silver.dims_agents`
AS

SELECT ROW_NUMBER() OVER(ORDER BY agent_name) AS agent_id, agent_name
FROM(
    SELECT DISTINCT agent AS agent_name
    FROM `{project_id}.silver.players_performance`
)
"""

client.query(query).result()

print(f"Silver table created: {project_id}.silver.dims_agents")

Silver table created: valorant-project-2026.silver.dims_agents


In [ ]:
query = f"""
SELECT
    'dims_tours' AS table_name,
    'tour_id_unique' AS test_name,
    'uniqueness' AS test_type,
    COUNT(*) - COUNT(DISTINCT tour_id) AS actual_value,
    0 AS expected_value,
    COUNT(*) = COUNT(DISTINCT tour_id) AS passed
FROM `{project_id}.silver.dims_tours`
UNION ALL
SELECT
    'dims_tours' AS table_name,
    'tour_id_null' AS test_name,
    'completeness' AS test_type,
    COUNTIF(tour_id IS NULL) AS actual_value,
    0 AS expected_value,
    COUNT(tour_id IS NULL) == 0 AS passed
FROM `{project_id}.silver.dims_tours`
UNION ALL
SELECT
    'dims_tours' AS table_name,
    'tour_id_unique' AS test_name,
    'uniqueness' AS test_type,
    COUNT(*) - COUNT(DISTINCT tour_id) AS actual_value,
    0 AS expected_value,
    COUNT(*) = COUNT(DISTINCT tour_id) AS passed
FROM `{project_id}.silver.dims_tours`
;
"""

result = client.query(query).result()
# for row in result:
#     status = "PASS" if row.passed else "FAIL"

#     print(f"[{status}] {row.test_name}")

c:\Users\ACER\miniconda3\envs\valorant\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,test_name,test_type,actual_value,expected_value,passed
0,dims_tours,tour_id_unique,uniqueness,0,0,True


In [ ]:
# Table Schema Check
table = client.get_table(
    f"{project_id}.silver.matches"
)

print("Rows: ", table.num_rows)

for field in table.schema:
    print(field.name, field.field_type)

Rows:  1259
tour_id INTEGER
match_id INTEGER
match_datetime DATETIME
match_date DATE
bracket STRING
home_name STRING
home_alias STRING
away_name STRING
away_alias STRING
bo STRING
patch FLOAT
home_score INTEGER
away_score INTEGER
home_h2h_win INTEGER
away_h2h_win INTEGER
home_h2h_score INTEGER
away_h2h_score INTEGER
home_n_last_match INTEGER
away_n_last_match INTEGER
home_n_last_wr FLOAT
away_n_last_wr FLOAT


In [37]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.gold.fact_matches`
PARTITION BY match_date
AS

WITH round_score AS (
    SELECT DISTINCT
        match_id,
        SUM(home_score) OVER (PARTITION BY match_id) AS home_total_round,
        SUM(away_score) OVER (PARTITION BY match_id) AS away_total_round
    FROM `{project_id}.silver.games_overview`
)
SELECT
    tour_id,
    m.match_id,
    match_date,
    match_datetime,
    bracket,
    t1.team_id AS home_team_id,
    t2.team_id AS away_team_id,
    bo,
    patch,
    home_score,
    away_score,
    ROUND((1.0 * home_score - away_score) / NULLIF(1.0 * home_score + away_score, 0), 2) AS score_diff_pct,
    r.home_total_round AS home_total_round,
    r.away_total_round AS away_total_round,
    ROUND((1.0 * home_total_round - away_total_round) / NULLIF(1.0 * home_total_round + away_total_round, 0), 2) AS match_round_diff_pct,
    home_h2h_win,
    away_h2h_win,
    ROUND((1.0 * home_h2h_win - away_h2h_win) / NULLIF(1.0 * home_h2h_win + away_h2h_win, 0)) AS h2h_win_pct,
    home_h2h_score,
    away_h2h_score,
    ROUND((1.0 * home_h2h_score - away_h2h_score) / NULLIF(1.0 * home_h2h_score + away_h2h_score, 0), 2) AS h2h_game_win_pct,
    home_n_last_match,
    away_n_last_match,
    home_n_last_wr,
    away_n_last_wr,
    
    CASE
        WHEN home_score > away_score
        THEN 1
        ELSE 0 
    END AS is_home_win
FROM `{project_id}.silver.matches` m
JOIN round_score r
ON r.match_id = m.match_id
JOIN `{project_id}.silver.dims_teams` t1
ON t1.team_name = m.home_name
JOIN `{project_id}.silver.dims_teams` t2
ON t2.team_name = m.away_name
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.fact_matches")

Gold table created: valorant-project-2026.gold.fact_matches


In [47]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.gold.fact_games`
PARTITION BY match_date
AS

SELECT
    tour_id,
    o.match_id,
    o.game_id,    
    o.match_date,
    o.match_datetime,
    f.home_team_id,
    f.away_team_id,
    m.map_id,
    game_duration,

    o.home_score,
    o.away_score,
    o.home_score + o.away_score AS total_round,
    ROUND((1.0 * o.home_score - o.away_score) / NULLIF(o.home_score + o.away_score, 0), 2) AS score_diff_ratio,

    home_atk_score,
    away_atk_score,
    ROUND((1.0 * home_atk_score / NULLIF(home_atk_score + away_def_score, 2)) / NULLIF(1.0 * away_atk_score / NULLIF(away_atk_score + home_def_score, 0), 0), 2) AS atk_wr_ratio,

    home_def_score,
    away_def_score,
    ROUND((1.0 * home_def_score / NULLIF(home_def_score + away_atk_score, 2)) / NULLIF(1.0 * away_def_score / NULLIF(away_def_score + home_atk_score, 0), 0), 2) AS def_wr_ratio,
    
    home_ot_score,
    away_ot_score,
    ROUND((1.0 * home_ot_score - away_ot_score) / NULLIF(home_ot_score + away_ot_score, 0), 2) AS ot_score_diff_ratio,

    e.home_pstl_win,
    e.away_pstl_win,
    e.home_pstl_win - e.away_pstl_win AS pstl_win_diff,

    e.home_eco_round,
    e.away_eco_round,
    e.home_eco_win,
    e.away_eco_win,
    ROUND((1.0 * e.home_eco_win / NULLIF(e.home_eco_round, 0)) - (1.0 * e.away_eco_win / NULLIF(e.away_eco_round, 0)), 2) AS eco_wr_diff,
    
    e.home_semi_eco_round,
    e.away_semi_eco_round,
    e.home_semi_eco_win,
    e.away_semi_eco_win,
    ROUND ((1.0 * e.home_semi_eco_win / NULLIF(e.home_semi_eco_round, 0)) - (1.0 * e.away_semi_eco_win / NULLIF(e.away_semi_eco_round, 0)), 2) AS semi_eco_wr_diff,
    
    e.home_semi_buy_round,
    e.away_semi_buy_round,
    e.home_semi_buy_win,
    e.away_semi_buy_win,
    ROUND((1.0 * e.home_semi_buy_win / NULLIF(e.home_semi_buy_round, 0)) - (1.0 * e.away_semi_buy_win / NULLIF(e.away_semi_buy_round, 0)), 2) AS semi_buy_wr_diff,
    
    e.home_full_buy_round,
    e.away_full_buy_round,
    e.home_full_buy_win,
    e.away_full_buy_win,
    ROUND((1.0 * e.home_full_buy_win / NULLIF(e.home_full_buy_round, 0)) - (1.0 * e.away_full_buy_win / NULLIF(e.away_full_buy_round, 0)), 2) AS full_buy_wr_diff,
    
    CASE
        WHEN o.home_score > o.away_score
        THEN 1
        ELSE 0 
    END AS is_home_win
FROM `{project_id}.silver.games_overview` o
JOIN `{project_id}.silver.games_economy` e
ON o.game_id = e.game_id
JOIN `{project_id}.gold.fact_matches` f
ON o.match_id = f.match_id
JOIN `{project_id}.silver.dims_maps` m
ON o.game_map = m.map_name
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.fact_games")

Gold table created: valorant-project-2026.gold.fact_games


In [ ]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.gold.fact_map_vetos`
AS

WITH non_decider AS(
SELECT
    match_id,
    m.map_id,
    t.team_id,
    action,
    ROW_NUMBER() OVER (PARTITION BY match_id, t.team_id, action) AS action_order
FROM `{project_id}.silver.map_vetos` v
JOIN `{project_id}.silver.dims_teams` t
ON v.team_name = t.team_alias
JOIN `{project_id}.silver.dims_maps` m
ON v.map_name = m.map_name
WHERE action != "decider"
),
team_list AS(
    SELECT
        match_id,
        home_team_id,
        away_team_id
    FROM `{project_id}.gold.fact_matches`
),
decider_home AS(
SELECT
    v.match_id,
    m.map_id,
    t.home_team_id AS team_id,
    action,
    NULL AS action_order
FROM `{project_id}.silver.map_vetos` v
JOIN team_list t
ON v.match_id = t.match_id
JOIN `{project_id}.silver.dims_maps` m
ON v.map_name = m.map_name
WHERE action = 'decider'
),
decider_away AS(
SELECT
    v.match_id,
    m.map_id,
    t.away_team_id AS team_id,
    action,
    NULL AS action_order
FROM `{project_id}.silver.map_vetos` v
JOIN team_list t
ON v.match_id = t.match_id
JOIN `{project_id}.silver.dims_maps` m
ON v.map_name = m.map_name
WHERE action = 'decider'
)

SELECT * FROM non_decider

UNION ALL

SELECT * FROM decider_home

UNION ALL

SELECT * FROM decider_away;
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.fact_map_vetos")

Gold table created: valorant-project-2026.gold.fact_map_vetos


In [40]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.team_game_performance`
AS

SELECT
    tour_id,
    match_id,
    game_id,
    map_id,
    home_team_id AS team_id,
    
    CASE
        WHEN home_ot_score + away_ot_score > 0
        THEN 1
        ELSE 0
    END AS is_ot,

    is_home_win AS is_win,
    game_duration,
    score_diff_ratio,
    
    ROUND(home_atk_score * 1.0 / NULLIF(home_atk_score + away_def_score, 0), 2) AS atk_wr,
    ROUND(home_def_score * 1.0 / NULLIF(home_def_score + away_atk_score, 0), 2) AS def_wr,
    ROUND(home_pstl_win * 1.0 / NULLIF(home_pstl_win + away_pstl_win, 0), 2) AS pstl_wr,
    ROUND(home_eco_win * 1.0 / NULLIF(home_eco_round, 0), 2) AS eco_wr,
    ROUND(home_semi_eco_win * 1.0 / NULLIF(home_semi_eco_round, 0), 2) AS semi_eco_wr,
    ROUND(home_semi_buy_win * 1.0 / NULLIF(home_semi_buy_round, 0), 2) AS semi_buy_wr,
    ROUND(home_full_buy_win * 1.0 / NULLIF(home_full_buy_round, 0), 2) AS full_buy_wr
FROM `{project_id}.gold.fact_games`
UNION ALL
SELECT
    tour_id,
    match_id,
    game_id,
    map_id,
    away_team_id AS team_id,
    
    CASE
        WHEN home_ot_score + away_ot_score > 0
        THEN 1
        ELSE 0
    END AS is_ot,

    CASE
        WHEN is_home_win = 1
        THEN 0
        ELSE 1
    END AS is_win,

    game_duration,
    score_diff_ratio * (-1.0) AS score_diff_ratio,
    
    ROUND(away_atk_score * 1.0 / NULLIF(home_def_score + away_atk_score, 0), 2),
    ROUND(away_def_score * 1.0 / NULLIF(home_atk_score + away_def_score, 0), 2),
    ROUND(away_pstl_win * 1.0 / NULLIF(home_pstl_win + away_pstl_win, 0), 2),
    ROUND(away_eco_win * 1.0 / NULLIF(away_eco_round, 0), 2),
    ROUND(away_semi_eco_win * 1.0 / NULLIF(away_semi_eco_round, 0), 2),
    ROUND(away_semi_buy_win * 1.0 / NULLIF(away_semi_buy_round, 0), 2),
    ROUND(away_full_buy_win * 1.0 / NULLIF(away_full_buy_round, 0), 2)
FROM `{project_id}.gold.fact_games`
"""

client.query(query).result()

print(f"Gold view created: {project_id}.gold.team_game_performance")

Gold view created: valorant-project-2026.gold.team_game_performance


In [ ]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.team_map_performance`
AS

WITH team_pick_ban AS(
    SELECT
        team_id,
        map_id,

        SUM(
            CASE
                WHEN action = 'pick'
                THEN 1
                ELSE 0
            END
        ) AS pick_count,

        SUM(
            CASE
                WHEN action = 'ban'
                THEN 1
                ELSE 0
            END
        ) AS ban_count
    FROM `{project_id}.gold.fact_map_vetos`
    GROUP BY
        team_id,
        map_id
),
team_map_count AS(
    SELECT
        team_id,
        map_id,
        COUNT(*) AS map_played,
        SUM(is_ot) AS ot_played,
        SUM(is_win) AS maps_win,
        ROUND(AVG(game_duration), 2) AS avg_game_duration,
        ROUND(AVG(score_diff_ratio), 2) AS avg_score_diff_ratio,
        ROUND(AVG(atk_wr), 2) AS avg_atk_wr,
        ROUND(AVG(def_wr), 2) AS avg_def_wr,
        ROUND(AVG(pstl_wr), 2) AS avg_pstl_wr,
        ROUND(AVG(eco_wr), 2) AS avg_eco_wr,
        ROUND(AVG(semi_eco_wr), 2) AS avg_semi_eco_wr,
        ROUND(AVG(semi_buy_wr), 2) AS avg_semi_buy_wr,
        ROUND(AVG(full_buy_wr), 2) AS avg_full_buy_wr
    FROM `{project_id}.gold.team_game_performance`
    GROUP BY
        team_id,
        map_id
),
team_count AS (
    SELECT
        team_id,
        COUNT(DISTINCT match_id) AS total_matches
    FROM `{project_id}.gold.team_game_performance`
    GROUP BY team_id
)

SELECT
    m.team_id,
    m.map_id,
    map_played,
    ot_played,
    maps_win,

    ROUND(1.0 * maps_win / map_played, 2) AS map_wr,

    p.pick_count,
    p.ban_count,

    ROUND(1.0 * p.pick_count / NULLIF(c.total_matches, 0), 2) AS pick_rate,
    ROUND(1.0 * p.ban_count / NULLIF(c.total_matches, 0), 2) AS ban_rate,
    ROUND(1.0 * p.pick_count / NULLIF(p.pick_count + p.ban_count, 0), 2) AS map_pick_preference,

    avg_game_duration,
    avg_score_diff_ratio,
    avg_atk_wr,
    avg_def_wr,
    avg_pstl_wr,
    avg_eco_wr,
    avg_semi_eco_wr,
    avg_semi_buy_wr,
    avg_full_buy_wr
FROM team_map_count m
LEFT JOIN team_pick_ban p
ON m.team_id = p.team_id
AND m.map_id = p.map_id
JOIN team_count c
ON m.team_id = c.team_id
"""

client.query(query).result()

print(f"Gold view created: {project_id}.gold.team_map_performance")

Gold view created: valorant-project-2026.gold.team_map_performance


In [56]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.map_performance`
AS

WITH team_pick_ban AS(
    SELECT
        map_id,

        SUM(
            CASE
                WHEN action = 'pick'
                THEN 1
                ELSE 0
            END
        ) AS pick_count,

        SUM(
            CASE
                WHEN action = 'ban'
                THEN 1
                ELSE 0
            END
        ) AS ban_count
    FROM `{project_id}.gold.fact_map_vetos`
    GROUP BY map_id
),
map_count AS(
    SELECT
        map_id,
        COUNT(DISTINCT match_id) AS map_played,
        SUM(is_ot) AS ot_played,
        ROUND(AVG(game_duration), 2) AS avg_game_duration
    FROM `{project_id}.gold.team_game_performance`
    GROUP BY map_id
),
match_count AS (
    SELECT
        map_id,
        SUM(total_round) AS total_round,
        SUM(home_atk_score + away_atk_score) AS atk_side_score,
        SUM(home_def_score + away_def_score) AS def_side_score,
        (SELECT COUNT(DISTINCT match_id) FROM `{project_id}.gold.fact_games`) AS total_matches
    FROM `{project_id}.gold.fact_games`
    GROUP BY map_id
)

SELECT
    m.map_id,
    map_played,
    ot_played,

    p.pick_count,
    p.ban_count,

    ROUND(1.0 * p.pick_count / NULLIF(c.total_matches, 0), 2) AS pick_rate,
    ROUND(1.0 * p.ban_count / NULLIF(c.total_matches, 0), 2) AS ban_rate,
    ROUND(1.0 * p.pick_count / NULLIF(p.pick_count + p.ban_count, 0), 2) AS map_pick_preference,
    
    avg_game_duration,
    ROUND(1.0 * c.atk_side_score / NULLIF(c.total_round, 0), 2) AS atk_side_ratio,
    ROUND(1.0 * c.def_side_score / NULLIF(c.total_round, 0), 2) AS def_side_ratio

FROM map_count m
LEFT JOIN team_pick_ban p
ON m.map_id = p.map_id
JOIN match_count c
ON m.map_id = c.map_id
"""

client.query(query).result()

print(f"Gold view created: {project_id}.gold.map_performance")

Gold view created: valorant-project-2026.gold.map_performance


In [32]:
query = f"""
CREATE OR REPLACE TABLE `{project_id}.gold.fact_players_performance`
AS

WITH statistics AS(
    SELECT
        tour_id,
        match_id,
        game_id,
        map_id,
        p.team_id,
        t.team_alias,
        is_win
    FROM `{project_id}.gold.team_game_performance` p
    JOIN `{project_id}.silver.dims_teams` t
    ON p.team_id = t.team_id
)
SELECT
    s.tour_id,
    s.match_id,
    pp.game_id,
    s.map_id,
    p.player_id,
    s.team_id,
    ag.agent_id,
    s.is_win,
    mod,
    r,
    acs,
    k,
    d,
    a,
    kd,
    kast,
    adr,
    hs,
    fk,
    fd,
    fkfd
FROM `{project_id}.silver.players_performance` pp
JOIN statistics s
ON pp.game_id = s.game_id
AND pp.team_alias = s.team_alias
JOIN `{project_id}.silver.dims_players` p
ON pp.player_name = p.player_name
JOIN `{project_id}.silver.dims_agents` ag
ON pp.agent = ag.agent_name
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.fact_players_performance")

Gold table created: valorant-project-2026.gold.fact_players_performance


In [33]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.player_agent_map_performance`
AS

SELECT
    player_id,
    agent_id,
    map_id,
    COUNT(map_id) AS agent_played,
    SUM(is_win) AS total_win,
    ROUND(1.0 * SUM(is_win) / COUNT(map_id), 2) AS agent_wr,
    ROUND(AVG(r), 2) AS avg_r,
    ROUND(AVG(acs), 2) AS avg_acs,
    ROUND(AVG(k), 2) AS avg_k,
    ROUND(AVG(d), 2) AS avg_d,
    ROUND(AVG(a), 2) AS avg_a,
    ROUND(AVG(kd), 2) AS avg_kd,
    ROUND(AVG(kast), 2) AS avg_kast,
    ROUND(AVG(adr), 2) AS avg_adr,
    ROUND(AVG(hs), 2) AS avg_hs,
    ROUND(AVG(fk), 2) AS avg_fk,
    ROUND(AVG(fd), 2) AS avg_fd,
    ROUND(AVG(fkfd), 2) AS avg_fkfd
FROM `{project_id}.gold.fact_players_performance`
WHERE mod = 'avg'
GROUP BY
    player_id,
    agent_id,
    map_id;
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.player_agent_map_performance")

Gold table created: valorant-project-2026.gold.player_agent_map_performance


In [4]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.agent_map_performance`
AS

SELECT
    agent_id,
    p.map_id,
    COUNT(DISTINCT game_id) AS presence_count,
    m.map_played,
    COUNT(agent_id) AS pick_count,
    SUM(is_win) AS total_win,
    ROUND(1.0 * COUNT(agent_id) / NULLIF(m.map_played, 0), 2) AS pick_rate,
    ROUND(1.0 * SUM(is_win) / NULLIF(COUNT(DISTINCT game_id), 0), 2) AS win_rate,
    ROUND(1.0 * COUNT(DISTINCT game_id) / NULLIF(m.map_played, 0), 2) AS presence_rate
FROM `{project_id}.gold.fact_players_performance` p
JOIN `{project_id}.gold.map_performance` m
ON p.map_id = m.map_id
WHERE mod = 'avg'
GROUP BY
    agent_id,
    p.map_id,
    m.map_played;
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.agent_map_performance")

Gold table created: valorant-project-2026.gold.agent_map_performance


In [35]:
query = f"""
CREATE OR REPLACE VIEW `{project_id}.gold.agent_performance`
AS

SELECT
    agent_id,
    COUNT(*) AS pick_count,
    COUNT(DISTINCT game_id) AS total_games,
    ROUND(1.0 * COUNT(*) / NULLIF(COUNT(DISTINCT game_id), 0), 2) AS pick_rate,
    ROUND(1.0 * SUM(is_win) / NULLIF(COUNT(*), 0), 2) AS win_rate,
FROM `{project_id}.gold.fact_players_performance`
WHERE mod = 'avg'
GROUP BY
    player_id,
    agent_id,
    map_id;
"""

client.query(query).result()

print(f"Gold table created: {project_id}.gold.agent_performance")

Gold table created: valorant-project-2026.gold.agent_performance
